# Wan 2.2 Long Video Generator (Ultimate Edition)

**Setup:** ComfyUI + WanVideoWrapper on Google Colab T4 (16GB VRAM)

## Features
- **Video Generation:** I2V, T2V, V2V modes via Wan 2.2 14B
- **Long Videos:** Up to 30 sec (721 frames @ 24fps)
- **Auto GPU Config:** Detects GPU and sets optimal parameters
- **Modular Downloads:** Pick only the models you need
- **Post-Processing:** Upscale (480p→1080p), frame interpolation (24→48fps), ffmpeg export
- **Batch Generation:** Queue multiple prompts via ComfyUI API
- **Google Drive:** Auto-save results
- **VRAM Monitor:** Real-time memory tracking
- **Telegram Bot:** Send prompts/photos in TG — get videos back automatically

## Quick Start
1. Run cells 1-5 in order
2. Open the Cloudflare tunnel URL
3. Load a workflow and hit Queue
4. Use Post-Processing cells for upscale/interpolation
5. (Optional) Run cells 11-12 to start Telegram bot

In [ ]:
#@title 1. Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB" if torch.cuda.is_available() else '')

In [ ]:
#@title 1.5 Auto GPU Config
import torch

GPU_PROFILES = {
    "T4":   {"blocks_to_swap": 20, "max_res": (832, 480),  "max_frames": 481, "dtype": "fp8",  "vram_gb": 16},
    "L4":   {"blocks_to_swap": 10, "max_res": (1024, 576), "max_frames": 601, "dtype": "fp8",  "vram_gb": 24},
    "A100": {"blocks_to_swap": 0,  "max_res": (1280, 720), "max_frames": 721, "dtype": "bf16", "vram_gb": 80},
    "V100": {"blocks_to_swap": 15, "max_res": (832, 480),  "max_frames": 481, "dtype": "fp16", "vram_gb": 16},
    "A10G": {"blocks_to_swap": 10, "max_res": (1024, 576), "max_frames": 601, "dtype": "fp8",  "vram_gb": 24},
}

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3 if torch.cuda.is_available() else 0

# Match GPU profile
GPU_CONFIG = GPU_PROFILES.get("T4")  # default
for key, profile in GPU_PROFILES.items():
    if key.lower() in gpu_name.lower():
        GPU_CONFIG = profile
        break

# Override with actual VRAM if higher (e.g. Colab Pro sometimes gives more)
if vram_gb > 20 and GPU_CONFIG["vram_gb"] <= 16:
    GPU_CONFIG["blocks_to_swap"] = max(GPU_CONFIG["blocks_to_swap"] - 5, 0)
    GPU_CONFIG["max_frames"] = 601

print(f"GPU: {gpu_name} ({vram_gb:.0f} GB)")
print(f"Profile: blocks_to_swap={GPU_CONFIG['blocks_to_swap']}, "
      f"max_res={GPU_CONFIG['max_res']}, max_frames={GPU_CONFIG['max_frames']}, "
      f"dtype={GPU_CONFIG['dtype']}")
print(f"\nUse these values in your workflow nodes.")

In [ ]:
#@title 2. Install ComfyUI
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI
!pip install -r requirements.txt -q

In [ ]:
#@title 3. Install Custom Nodes
%cd /content/ComfyUI/custom_nodes

# Core video nodes
!git clone https://github.com/kijai/ComfyUI-WanVideoWrapper.git
!git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git
!git clone https://github.com/kijai/ComfyUI-KJNodes.git

# Frame interpolation (RIFE)
!git clone https://github.com/Fannovel16/ComfyUI-Frame-Interpolation.git

# Face detection, segmentation, regional upscale
!git clone https://github.com/ltdrdata/ComfyUI-Impact-Pack.git

# Install dependencies
import glob
for req in glob.glob("*/requirements.txt"):
    print(f"Installing {req}...")
    !pip install -r {req} -q

# Impact Pack has a separate install script
%cd /content/ComfyUI/custom_nodes/ComfyUI-Impact-Pack
!python install.py -q 2>/dev/null || true
%cd /content/ComfyUI/custom_nodes

print("\nAll custom nodes installed!")

In [ ]:
#@title 3.5 Install Sage Attention (~30% faster inference)
#@markdown Optimized attention kernel. Skip if install fails — everything still works without it.
!pip install sageattention -q 2>/dev/null || echo "SageAttention not available for this GPU, skipping (not critical)"
print("Sage Attention setup complete.")

In [ ]:
#@title 4. Download Models (Modular)
import os
import ipywidgets as widgets
from IPython.display import display

models_dir = "/content/ComfyUI/models"

# Model registry: (checkbox_label, size, target_path, url)
MODEL_REGISTRY = {
    "base_diffusion": {
        "label": "Wan 2.2 T2V 14B fp8 (REQUIRED)",
        "size": "~9.5 GB",
        "path": f"{models_dir}/diffusion_models/Wan2_2-T2V-A14B-LOW_fp8_e4m3fn_scaled_KJ.safetensors",
        "url": "https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/Wan2_2-T2V-A14B-LOW_fp8_e4m3fn_scaled_KJ.safetensors",
        "default": True,
    },
    "vace_module": {
        "label": "VACE Module 14B (long video)",
        "size": "~5 GB",
        "path": f"{models_dir}/diffusion_models/Wan2_2_Fun_VACE_module_A14B_LOW_bf16.safetensors",
        "url": "https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/Wan2_2_Fun_VACE_module_A14B_LOW_bf16.safetensors",
        "default": True,
    },
    "text_encoder": {
        "label": "UMT5 XXL Text Encoder fp8 (REQUIRED)",
        "size": "~4.9 GB",
        "path": f"{models_dir}/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
        "url": "https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
        "default": True,
    },
    "vae": {
        "label": "Wan 2.2 VAE (REQUIRED)",
        "size": "~200 MB",
        "path": f"{models_dir}/vae/wan2.2_vae.safetensors",
        "url": "https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/wan2.2_vae.safetensors",
        "default": True,
    },
    "upscale_realesrgan": {
        "label": "RealESRGAN x4plus (video upscale)",
        "size": "~64 MB",
        "path": f"{models_dir}/upscale_models/RealESRGAN_x4plus.pth",
        "url": "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth",
        "default": True,
    },
    "rife_model": {
        "label": "RIFE v4.6 (frame interpolation)",
        "size": "~115 MB",
        "path": f"{models_dir}/custom_nodes/ComfyUI-Frame-Interpolation/ckpts/rife/",
        "url": "AUTO",  # downloaded by the node itself on first use
        "default": False,
    },
}

print("Select models to download:\n")
checkboxes = {}
for key, model in MODEL_REGISTRY.items():
    cb = widgets.Checkbox(
        value=model["default"],
        description=f'{model["label"]} ({model["size"]})',
        style={"description_width": "initial"},
        layout=widgets.Layout(width="600px"),
    )
    checkboxes[key] = cb
    display(cb)

download_btn = widgets.Button(description="Download Selected", button_style="success")
output = widgets.Output()
display(download_btn, output)

def on_download(btn):
    with output:
        output.clear_output()
        for key, cb in checkboxes.items():
            if not cb.value:
                continue
            model = MODEL_REGISTRY[key]
            if model["url"] == "AUTO":
                print(f"[skip] {model['label']} — downloads automatically on first use")
                continue
            target_dir = os.path.dirname(model["path"])
            os.makedirs(target_dir, exist_ok=True)
            if os.path.exists(model["path"]):
                print(f"[exists] {model['label']}")
                continue
            print(f"[downloading] {model['label']}...")
            !wget -q --show-progress -O "{model['path']}" "{model['url']}"
        print("\nDone! Selected models downloaded.")
        !du -sh {models_dir}/diffusion_models/* {models_dir}/text_encoders/* {models_dir}/vae/* {models_dir}/upscale_models/* 2>/dev/null || true

download_btn.on_click(on_download)

In [ ]:
#@title 4.5 Download Workflow
import os
os.makedirs("/content/ComfyUI/user/default/workflows", exist_ok=True)
!wget -q -O /content/ComfyUI/user/default/workflows/video_wan_long_ultimate.json \
  "https://gist.githubusercontent.com/russiankendricklamar/ea1ab4a53e1be1b8401e5031bcdae4f3/raw/video_wan_long_ultimate.json"
print("Workflow downloaded!")

In [ ]:
#@title 5. Launch ComfyUI with Cloudflare Tunnel
import subprocess
import time

# Install cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# Start ComfyUI in background
comfy_proc = subprocess.Popen(
    ['python', 'main.py', '--listen', '0.0.0.0', '--port', '8188'],
    cwd='/content/ComfyUI',
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

print("Starting ComfyUI... waiting 30s for startup")
time.sleep(30)

# Start cloudflare tunnel
tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8188'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

time.sleep(5)

# Find tunnel URL
import re
for _ in range(10):
    line = tunnel_proc.stdout.readline().decode()
    match = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
    if match:
        url = match.group(0)
        print(f"\n{'='*60}")
        print(f"ComfyUI is ready!")
        print(f"Open this URL: {url}")
        print(f"{'='*60}")
        print(f"\nLoad workflow: video_wan_long_ultimate")
        break
else:
    print("Tunnel URL not found yet. Check 'cloudflared' output manually.")
    print("ComfyUI should be running on port 8188")

## Usage

### Basic Workflow
1. Open the Cloudflare URL from cell 5
2. Load workflow: **Menu -> Load -> `video_wan_long_ultimate`**
3. Upload start image to `LoadImage` node
4. Edit prompt in `Text Conditioning` node
5. Hit **Queue Prompt**

### Duration Control
| Frames | Duration |
|--------|----------|
| 81 | 3.4 sec |
| 241 | 10 sec |
| 481 | 20 sec |
| 721 | 30 sec |

### Resolution
| Size | Aspect |
|------|--------|
| 832x480 | 16:9 landscape |
| 480x832 | 9:16 portrait (Reels) |
| 608x1080 | 9:16 HD portrait |

### Post-Processing (cell 7)
- **Upscale:** RealESRGAN 4x (480p -> 1080p)
- **Interpolation:** RIFE/minterpolate (24fps -> 48/60fps)
- **Export:** MP4, WebM, or GIF with optional audio
- Auto-detects latest generated video

### Batch Generation (cell 8)
- Put one prompt per line
- Sends each to ComfyUI API sequentially
- Waits for completion before next

### Google Drive (cell 6)
- Auto-mounts Drive on run
- Call `sync_to_drive()` to copy new outputs

### VRAM Monitor (cell 9)
- Single check or live mode with auto-refresh
- Shows allocated/reserved/free VRAM with progress bar

### Telegram Bot (cells 11-12)
1. Get a bot token from @BotFather in Telegram
2. Run cell 11 — paste token, set allowed user IDs
3. Run cell 12 — bot starts!
4. Open your bot in Telegram, press Start
5. Pick "Написать -> Видео" or "Картинка -> Видео"
6. Choose duration and format
7. Wait — bot sends the finished video right in the chat
8. Video also saved to Google Drive automatically

In [ ]:
#@title 6. Mount Google Drive (Auto-Save)
from google.colab import drive
import shutil
import os

drive.mount("/content/drive")

DRIVE_OUTPUT = "/content/drive/MyDrive/ComfyUI_Output"
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

COMFY_OUTPUT = "/content/ComfyUI/output"

def sync_to_drive():
    """Copy all new files from ComfyUI output to Google Drive."""
    if not os.path.exists(COMFY_OUTPUT):
        print("No output directory yet. Generate a video first.")
        return
    files = os.listdir(COMFY_OUTPUT)
    if not files:
        print("No files to sync.")
        return
    copied = 0
    for f in files:
        src = os.path.join(COMFY_OUTPUT, f)
        dst = os.path.join(DRIVE_OUTPUT, f)
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
            copied += 1
            print(f"  Copied: {f}")
    print(f"\nSynced {copied} new file(s) to {DRIVE_OUTPUT}")

print(f"Drive mounted. Output will be saved to: {DRIVE_OUTPUT}")
print("Run sync_to_drive() anytime to copy new results to Drive.")

In [ ]:
#@title 7. Post-Processing Pipeline (Upscale + Interpolation + Export)
import os
import subprocess

COMFY_OUTPUT = "/content/ComfyUI/output"
PROCESSED_DIR = "/content/ComfyUI/output_processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

#@markdown ### Settings
input_video = "" #@param {type:"string"}
enable_upscale = True #@param {type:"boolean"}
upscale_factor = 4 #@param [2, 4] {type:"raw"}
enable_interpolation = False #@param {type:"boolean"}
target_fps = 48 #@param [48, 60] {type:"raw"}
add_audio = "" #@param {type:"string"}
output_format = "mp4" #@param ["mp4", "webm", "gif"]

def parse_fps(probe_output):
    """Safely parse fps from ffprobe output like '24/1' or '30000/1001'."""
    try:
        text = probe_output.strip()
        if "/" in text:
            num, den = text.split("/")
            return int(num) / int(den)
        return float(text)
    except (ValueError, ZeroDivisionError):
        return 24

# Auto-detect latest video if input not specified
if not input_video:
    videos = sorted(
        [f for f in os.listdir(COMFY_OUTPUT) if f.endswith((".mp4", ".webm", ".png"))],
        key=lambda f: os.path.getmtime(os.path.join(COMFY_OUTPUT, f)),
        reverse=True,
    )
    if videos:
        input_video = os.path.join(COMFY_OUTPUT, videos[0])
        print(f"Auto-detected: {input_video}")
    else:
        print("No videos found in output. Generate a video first.")

if input_video and os.path.exists(input_video):
    basename = os.path.splitext(os.path.basename(input_video))[0]
    current = input_video

    # Step 1: Upscale with RealESRGAN
    if enable_upscale:
        print("\n[1/3] Upscaling with RealESRGAN...")
        !pip install realesrgan -q 2>/dev/null
        upscaled = os.path.join(PROCESSED_DIR, f"{basename}_upscaled.mp4")

        # Extract frames, upscale, reassemble
        frames_dir = os.path.join(PROCESSED_DIR, "frames_in")
        frames_up_dir = os.path.join(PROCESSED_DIR, "frames_up")
        os.makedirs(frames_dir, exist_ok=True)
        os.makedirs(frames_up_dir, exist_ok=True)

        !ffmpeg -y -i "{current}" -qscale:v 2 "{frames_dir}/%06d.png" -loglevel error
        !python -m realesrgan -i "{frames_dir}" -o "{frames_up_dir}" -n RealESRGAN_x4plus -s {upscale_factor} --suffix "" 2>/dev/null || \
            !realesrgan-ncnn-vulkan -i "{frames_dir}" -o "{frames_up_dir}" -n realesrgan-x4plus -s {upscale_factor} 2>/dev/null

        # Get original fps safely
        fps_str = !ffprobe -v error -select_streams v -of csv=p=0 -show_entries stream=r_frame_rate "{current}"
        fps = parse_fps(fps_str[0]) if fps_str else 24

        !ffmpeg -y -framerate {fps} -i "{frames_up_dir}/%06d.png" -c:v libx264 -pix_fmt yuv420p "{upscaled}" -loglevel error
        current = upscaled
        print(f"  Upscaled to: {upscaled}")

        # Cleanup temp frames
        !rm -rf "{frames_dir}" "{frames_up_dir}"

    # Step 2: Frame interpolation with RIFE
    if enable_interpolation:
        print("\n[2/3] Interpolating frames with RIFE...")
        interpolated = os.path.join(PROCESSED_DIR, f"{basename}_interpolated.mp4")
        !pip install rife-ncnn-vulkan-python -q 2>/dev/null || true

        # Use ffmpeg + RIFE via minterpolate as fallback
        !ffmpeg -y -i "{current}" -vf "minterpolate=fps={target_fps}:mi_mode=mci:mc_mode=aobmc:me_mode=bidir:vsbmc=1" \
            -c:v libx264 -pix_fmt yuv420p "{interpolated}" -loglevel error
        current = interpolated
        print(f"  Interpolated to {target_fps}fps: {interpolated}")

    # Step 3: Final export with ffmpeg
    print("\n[3/3] Final export...")
    final = os.path.join(PROCESSED_DIR, f"{basename}_final.{output_format}")

    audio_flag = f'-i "{add_audio}" -c:a aac -shortest' if add_audio and os.path.exists(add_audio) else "-an"

    if output_format == "gif":
        !ffmpeg -y -i "{current}" -vf "fps=15,scale=480:-1:flags=lanczos,split[s0][s1];[s0]palettegen[p];[s1][p]paletteuse" \
            "{final}" -loglevel error
    elif output_format == "webm":
        !ffmpeg -y -i "{current}" {audio_flag} -c:v libvpx-vp9 -crf 30 -b:v 0 "{final}" -loglevel error
    else:
        !ffmpeg -y -i "{current}" {audio_flag} -c:v libx264 -crf 18 -preset slow -pix_fmt yuv420p "{final}" -loglevel error

    print(f"\nDone! Final video: {final}")
    file_size = os.path.getsize(final) / (1024 * 1024)
    print(f"File size: {file_size:.1f} MB")

    # Auto sync to Drive
    if os.path.exists("/content/drive/MyDrive/ComfyUI_Output"):
        import shutil
        shutil.copy2(final, f"/content/drive/MyDrive/ComfyUI_Output/{os.path.basename(final)}")
        print(f"Saved to Google Drive!")

In [ ]:
#@title 8. Batch Generation (ComfyUI API)
import json
import urllib.request
import urllib.parse
import time

COMFYUI_URL = "http://localhost:8188"

#@markdown ### Prompts (one per line)
prompts_text = """a beautiful woman walking through a cherry blossom garden, cinematic, slow motion
a futuristic city skyline at sunset, neon lights, flying cars, cyberpunk style
ocean waves crashing on rocks, golden hour, dramatic slow motion""" #@param {type:"string"}

#@markdown ### Settings
num_frames = 241 #@param {type:"integer"}
width = 832 #@param {type:"integer"}
height = 480 #@param {type:"integer"}
steps = 30 #@param {type:"integer"}
cfg = 6.0 #@param {type:"number"}

def get_workflow():
    """Load the workflow JSON template."""
    workflow_path = "/content/ComfyUI/user/default/workflows/video_wan_long_ultimate.json"
    with open(workflow_path, "r") as f:
        return json.load(f)

def queue_prompt(workflow_json):
    """Send a workflow to ComfyUI queue."""
    data = json.dumps({"prompt": workflow_json}).encode("utf-8")
    req = urllib.request.Request(
        f"{COMFYUI_URL}/prompt",
        data=data,
        headers={"Content-Type": "application/json"},
    )
    resp = urllib.request.urlopen(req)
    return json.loads(resp.read())

def get_queue_status():
    """Check current queue status."""
    resp = urllib.request.urlopen(f"{COMFYUI_URL}/queue")
    data = json.loads(resp.read())
    running = len(data.get("queue_running", []))
    pending = len(data.get("queue_pending", []))
    return running, pending

def wait_for_completion(prompt_id, timeout=3600):
    """Wait for a specific prompt to complete."""
    start = time.time()
    while time.time() - start < timeout:
        resp = urllib.request.urlopen(f"{COMFYUI_URL}/history/{prompt_id}")
        history = json.loads(resp.read())
        if prompt_id in history:
            return history[prompt_id]
        time.sleep(5)
    return None

prompts = [p.strip() for p in prompts_text.strip().split("\n") if p.strip()]
print(f"Queuing {len(prompts)} prompts...\n")

for i, prompt in enumerate(prompts):
    print(f"[{i+1}/{len(prompts)}] {prompt[:80]}...")

    try:
        workflow = get_workflow()

        # Update prompt text in workflow nodes
        # (node IDs may vary — this searches for text input nodes)
        for node_id, node in workflow.items():
            if isinstance(node, dict):
                inputs = node.get("inputs", {})
                # Look for text/prompt fields
                for key in ["text", "prompt", "positive"]:
                    if key in inputs and isinstance(inputs[key], str):
                        inputs[key] = prompt

        result = queue_prompt(workflow)
        prompt_id = result.get("prompt_id", "unknown")
        print(f"  Queued: {prompt_id}")

        # Wait for completion
        print(f"  Generating...", end="", flush=True)
        history = wait_for_completion(prompt_id)
        if history:
            print(f" Done!")
        else:
            print(f" Timeout (still running in background)")

    except Exception as e:
        print(f"  Error: {e}")

print(f"\nBatch complete! Check /content/ComfyUI/output/ for results.")
print("Run sync_to_drive() to copy results to Google Drive.")

In [ ]:
#@title 9. VRAM Monitor
import torch
import time
from IPython.display import clear_output

def vram_status():
    """Print current VRAM usage."""
    if not torch.cuda.is_available():
        print("No CUDA GPU available")
        return
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    free = total - reserved
    pct = (reserved / total) * 100

    bar_len = 40
    filled = int(bar_len * reserved / total)
    bar = "=" * filled + "-" * (bar_len - filled)

    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: [{bar}] {pct:.0f}%")
    print(f"  Allocated: {allocated:.1f} GB")
    print(f"  Reserved:  {reserved:.1f} GB")
    print(f"  Free:      {free:.1f} GB / {total:.1f} GB total")

#@markdown ### Mode
live_monitor = False #@param {type:"boolean"}
refresh_sec = 5 #@param {type:"integer"}

if live_monitor:
    try:
        while True:
            clear_output(wait=True)
            vram_status()
            time.sleep(refresh_sec)
    except KeyboardInterrupt:
        print("Monitor stopped.")
else:
    vram_status()

In [ ]:
#@title 10. Optimization Toolkit (Speed + VRAM + Quality)
import os, json, shutil, torch, gc

COMFY_OUTPUT = "/content/ComfyUI/output"
DRIVE_CACHE = "/content/drive/MyDrive/ComfyUI_ModelCache"
models_dir = "/content/ComfyUI/models"

# ========== 1. MODEL CACHE ON DRIVE ==========
#@markdown ### Cache models on Google Drive (skip 20GB download next session)
enable_model_cache = True #@param {type:"boolean"}

def cache_models_to_drive():
    """Copy downloaded models to Drive for reuse."""
    os.makedirs(DRIVE_CACHE, exist_ok=True)
    for subdir in ["diffusion_models", "text_encoders", "vae", "upscale_models"]:
        src = os.path.join(models_dir, subdir)
        dst = os.path.join(DRIVE_CACHE, subdir)
        if not os.path.exists(src):
            continue
        os.makedirs(dst, exist_ok=True)
        for f in os.listdir(src):
            src_f, dst_f = os.path.join(src, f), os.path.join(dst, f)
            if os.path.isfile(src_f) and not os.path.exists(dst_f):
                print(f"  Caching {f}...")
                shutil.copy2(src_f, dst_f)
    print("Models cached to Drive!")

def restore_models_from_drive():
    """Restore models from Drive cache (instant, no download)."""
    if not os.path.exists(DRIVE_CACHE):
        print("No cache found on Drive.")
        return False
    restored = 0
    for subdir in ["diffusion_models", "text_encoders", "vae", "upscale_models"]:
        src = os.path.join(DRIVE_CACHE, subdir)
        dst = os.path.join(models_dir, subdir)
        if not os.path.exists(src):
            continue
        os.makedirs(dst, exist_ok=True)
        for f in os.listdir(src):
            src_f, dst_f = os.path.join(src, f), os.path.join(dst, f)
            if os.path.isfile(src_f) and not os.path.exists(dst_f):
                print(f"  Restoring {f}...")
                os.symlink(src_f, dst_f)
                restored += 1
    if restored:
        print(f"Restored {restored} models from Drive cache (symlinked)!")
    else:
        print("All models already present.")
    return restored > 0

if enable_model_cache and os.path.exists("/content/drive"):
    restore_models_from_drive()

# ========== 2. TEACACHE CONFIG ==========
#@markdown ### TeaCache (2-3x faster generation)
#@markdown Enable in WanVideoWrapper node: set `enable_teacache = true`
teacache_threshold = 0.3 #@param {type:"number"}
print(f"\nTeaCache: Set threshold={teacache_threshold} in WanVideoWrapper node")
print("  Lower = faster but less quality. Range: 0.1 (fast) - 0.5 (quality)")

# ========== 3. OPTIMAL GENERATION SETTINGS ==========
#@markdown ### Recommended workflow settings for T4
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3 if torch.cuda.is_available() else 16

print(f"\n{'='*50}")
print(f"OPTIMAL SETTINGS FOR YOUR GPU ({vram_gb:.0f}GB VRAM)")
print(f"{'='*50}")

if vram_gb <= 16:
    print("""
WanVideoWrapper node:
  blocks_to_swap: 20 (increase to 30 if OOM)
  enable_teacache: true
  teacache_threshold: 0.3
  attention_mode: sage (if installed) or sdpa

VACEEncode node:
  width: 832, height: 480
  num_frames: 241 (10s) or 481 (20s)

Sampler:
  steps: 20 (not 30 — Flow Match scheduler converges fast)
  cfg: 5.0-6.0
  scheduler: flow_match
  
VAE Decode:
  enable_tiling: true (saves ~2GB VRAM)
  tile_size: 256
""")
elif vram_gb <= 24:
    print("""
WanVideoWrapper node:
  blocks_to_swap: 10
  enable_teacache: true
  teacache_threshold: 0.3
  attention_mode: sage (if installed) or sdpa

VACEEncode: 1024x576, up to 601 frames
Sampler: steps=25, cfg=5.5, scheduler=flow_match
VAE Decode: enable_tiling=true, tile_size=512
""")
else:
    print("""
WanVideoWrapper node:
  blocks_to_swap: 0
  enable_teacache: true
  teacache_threshold: 0.2
  attention_mode: sage

VACEEncode: 1280x720, up to 721 frames  
Sampler: steps=30, cfg=5.0, scheduler=flow_match
VAE Decode: enable_tiling=false
""")

# ========== 4. NEGATIVE PROMPT TEMPLATES ==========
print(f"\n{'='*50}")
print("NEGATIVE PROMPT TEMPLATES")
print(f"{'='*50}")
print("""
General (copy to negative prompt):
  "blurry, low quality, distorted, deformed, ugly, bad anatomy, watermark, text, logo, oversaturated"

Realistic style:
  "cartoon, anime, illustration, painting, drawing, 3d render, cgi, unrealistic, plastic skin"

Anime style:
  "photo, realistic, photograph, 3d, ugly, deformed, noisy, blurry, low contrast"

Portrait/Face:
  "deformed face, extra fingers, mutated hands, bad proportions, extra limbs, cross-eyed, blurry face"
""")

# ========== 5. CFG RESCALE ==========
print(f"{'='*50}")
print("CFG RESCALE (anti-oversaturation)")
print(f"{'='*50}")
print("""
If colors look oversaturated at high CFG (>6):
  Add a 'RescaleCFG' node between sampler and model
  Set rescale_multiplier: 0.7
  This preserves detail while reducing color burn
""")

# ========== 6. OOM AUTO-RECOVERY ==========
def clear_vram():
    """Emergency VRAM cleanup."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    print("VRAM cleared. Retry with lower settings if OOM persists.")
    print("  Try: blocks_to_swap=30, resolution=480x320, num_frames=81")

print(f"{'='*50}")
print("OOM RECOVERY: Run clear_vram() if you get out-of-memory errors")
print(f"{'='*50}")

if enable_model_cache and os.path.exists("/content/drive"):
    print("\nTip: Run cache_models_to_drive() after downloading models")
    print("     Next session: models restore instantly via symlinks!")

In [ ]:
#@title 11. Telegram Bot Config
import getpass

#@markdown ### Bot Setup
#@markdown 1. Open @BotFather in Telegram
#@markdown 2. Send /newbot, pick a name
#@markdown 3. Copy the token and paste below

BOT_TOKEN = getpass.getpass("Paste your Telegram bot token: ")

#@markdown ### Who can use the bot?
#@markdown Enter Telegram user IDs separated by commas.
#@markdown To find your ID: message @userinfobot in Telegram.
ALLOWED_USERS = "123456789, 987654321" #@param {type:"string"}
ALLOWED_USER_IDS = [int(uid.strip()) for uid in ALLOWED_USERS.split(",") if uid.strip()]

#@markdown ### Defaults
MAX_QUEUE_SIZE = 10 #@param {type:"integer"}
DEFAULT_STEPS = 20 #@param {type:"integer"}
DEFAULT_CFG = 5.5 #@param {type:"number"}
ENABLE_UPSCALE = True #@param {type:"boolean"}
ENABLE_INTERPOLATION = True #@param {type:"boolean"}
TARGET_FPS = 48 #@param {type:"integer"}

COMFYUI_URL = "http://localhost:8188"
COMFY_OUTPUT = "/content/ComfyUI/output"
PROCESSED_DIR = "/content/ComfyUI/output_processed"
DRIVE_OUTPUT = "/content/drive/MyDrive/ComfyUI_Output"
WORKFLOW_PATH = "/content/ComfyUI/user/default/workflows/video_wan_long_ultimate.json"

# Duration/resolution mappings (hidden from user)
DURATION_MAP = {
    "short": {"frames": 81, "label": "Короткое (3 сек)", "est_min": 3},
    "medium": {"frames": 241, "label": "Среднее (10 сек)", "est_min": 5},
    "long": {"frames": 481, "label": "Длинное (20 сек)", "est_min": 10},
}

ORIENTATION_MAP = {
    "horizontal": {"width": 832, "height": 480, "label": "Горизонтальное"},
    "vertical": {"width": 480, "height": 832, "label": "Вертикальное"},
}

print(f"Bot token set: {'Yes' if BOT_TOKEN else 'No'}")
print(f"Allowed users: {ALLOWED_USER_IDS}")
print(f"Max queue: {MAX_QUEUE_SIZE}")
print(f"\nRun the next cell to start the bot!")

In [ ]:
#@title 12. Start Telegram Bot
#@markdown Run this cell to launch the bot. It will keep running until you stop it.
#@markdown
#@markdown **How to use:** Open your bot in Telegram and press Start!

import os
import json
import time
import copy
import asyncio
import shutil
import subprocess
import urllib.request
import logging
from collections import deque

# Install python-telegram-bot
!pip install python-telegram-bot==21.6 -q

from telegram import Update, InlineKeyboardButton, InlineKeyboardMarkup
from telegram.ext import (
    Application,
    CommandHandler,
    CallbackQueryHandler,
    MessageHandler,
    ConversationHandler,
    ContextTypes,
    filters,
)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("VideoBot")

# ==================== INTRO VIDEO ====================
INTRO_VIDEO_URL = "https://youtu.be/SxnjiI-3GsE"

# ==================== STATES ====================
CHOOSE_MODE, WAIT_PHOTO, WAIT_PROMPT, CHOOSE_DURATION, CHOOSE_ORIENTATION = range(5)

# ==================== JOB QUEUE ====================
job_queue = asyncio.Queue()
generation_times = deque(maxlen=20)

def estimate_wait(position):
    """Estimate wait time based on past generations."""
    if not generation_times:
        return position * 5
    avg = sum(generation_times) / len(generation_times)
    return int(position * avg)

# ==================== ACCESS CONTROL ====================
def authorized(func):
    """Decorator: only allow whitelisted users."""
    async def wrapper(update: Update, context: ContextTypes.DEFAULT_TYPE):
        user_id = update.effective_user.id
        if ALLOWED_USER_IDS and user_id not in ALLOWED_USER_IDS:
            text = "У тебя нет доступа к этому боту."
            if update.callback_query:
                await update.callback_query.answer(text, show_alert=True)
            else:
                await update.message.reply_text(text)
            return ConversationHandler.END
        return await func(update, context)
    return wrapper

# ==================== HANDLERS ====================

@authorized
async def start(update: Update, context: ContextTypes.DEFAULT_TYPE) -> int:
    """Entry point — send intro video and show main menu."""
    context.user_data.clear()

    # Send intro video (Jarvis activation scene)
    await update.message.reply_video(
        video=INTRO_VIDEO_URL,
        caption="Система активирована."
    )

    keyboard = [
        [InlineKeyboardButton("Написать -> Видео", callback_data="mode_t2v")],
        [InlineKeyboardButton("Картинка -> Видео", callback_data="mode_i2v")],
    ]
    await update.message.reply_text(
        "Привет! Я делаю видео из текста и картинок.\n\n"
        "Что хочешь сделать?\n"
        "(Отправь /cancel чтобы отменить в любой момент)",
        reply_markup=InlineKeyboardMarkup(keyboard),
    )
    return CHOOSE_MODE

@authorized
async def restart_entry(update: Update, context: ContextTypes.DEFAULT_TYPE) -> int:
    """Handle 'Сделать ещё' / 'Попробовать снова' — re-enter conversation."""
    query = update.callback_query
    await query.answer()
    context.user_data.clear()
    keyboard = [
        [InlineKeyboardButton("Написать -> Видео", callback_data="mode_t2v")],
        [InlineKeyboardButton("Картинка -> Видео", callback_data="mode_i2v")],
    ]
    await query.edit_message_text(
        "Что хочешь сделать?",
        reply_markup=InlineKeyboardMarkup(keyboard),
    )
    return CHOOSE_MODE

@authorized
async def choose_mode(update: Update, context: ContextTypes.DEFAULT_TYPE) -> int:
    """User picked T2V or I2V."""
    query = update.callback_query
    await query.answer()
    mode = query.data
    context.user_data["mode"] = mode

    if mode == "mode_i2v":
        await query.edit_message_text(
            "Отправь картинку, которую хочешь оживить."
        )
        return WAIT_PHOTO
    else:
        await query.edit_message_text(
            "Напиши, что должно быть на видео.\n\n"
            "Например: кот летит в космосе на ракете"
        )
        return WAIT_PROMPT

@authorized
async def receive_photo(update: Update, context: ContextTypes.DEFAULT_TYPE) -> int:
    """User sent a photo for I2V."""
    photo = update.message.photo[-1]
    file = await photo.get_file()
    photo_path = f"/content/ComfyUI/input/tg_{update.effective_user.id}_{int(time.time())}.jpg"
    os.makedirs("/content/ComfyUI/input", exist_ok=True)
    await file.download_to_drive(photo_path)

    if not os.path.exists(photo_path):
        await update.message.reply_text("Не удалось скачать фото. Попробуй ещё раз.")
        return WAIT_PHOTO

    context.user_data["image_path"] = photo_path

    await update.message.reply_text(
        "Отлично! Теперь напиши, что должно происходить на видео.\n\n"
        "Например: девушка поворачивает голову и улыбается"
    )
    return WAIT_PROMPT

@authorized
async def receive_prompt(update: Update, context: ContextTypes.DEFAULT_TYPE) -> int:
    """User sent the text prompt."""
    context.user_data["prompt"] = update.message.text

    keyboard = [
        [InlineKeyboardButton("Короткое (3 сек)", callback_data="dur_short")],
        [InlineKeyboardButton("Среднее (10 сек)", callback_data="dur_medium")],
        [InlineKeyboardButton("Длинное (20 сек)", callback_data="dur_long")],
    ]
    await update.message.reply_text(
        "Как долго видео?",
        reply_markup=InlineKeyboardMarkup(keyboard),
    )
    return CHOOSE_DURATION

@authorized
async def choose_duration(update: Update, context: ContextTypes.DEFAULT_TYPE) -> int:
    """User picked duration."""
    query = update.callback_query
    await query.answer()
    dur_key = query.data.replace("dur_", "")
    context.user_data["duration"] = dur_key

    keyboard = [
        [InlineKeyboardButton("Горизонтальное (YouTube)", callback_data="orient_horizontal")],
        [InlineKeyboardButton("Вертикальное (Reels/TikTok)", callback_data="orient_vertical")],
    ]
    await query.edit_message_text(
        "Какой формат?",
        reply_markup=InlineKeyboardMarkup(keyboard),
    )
    return CHOOSE_ORIENTATION

@authorized
async def choose_orientation(update: Update, context: ContextTypes.DEFAULT_TYPE) -> int:
    """User picked orientation — submit job to queue."""
    query = update.callback_query
    await query.answer()
    orient_key = query.data.replace("orient_", "")
    context.user_data["orientation"] = orient_key

    if job_queue.qsize() >= MAX_QUEUE_SIZE:
        await query.edit_message_text(
            "Очередь полная, попробуй чуть позже.\n\n"
            "Нажми /start чтобы попробовать снова."
        )
        return ConversationHandler.END

    dur = DURATION_MAP[context.user_data["duration"]]
    orient = ORIENTATION_MAP[orient_key]
    job = {
        "user_id": update.effective_user.id,
        "chat_id": update.effective_chat.id,
        "mode": context.user_data["mode"],
        "prompt": context.user_data["prompt"],
        "image_path": context.user_data.get("image_path"),
        "frames": dur["frames"],
        "width": orient["width"],
        "height": orient["height"],
        "submitted_at": time.time(),
    }

    position = job_queue.qsize() + 1
    est = estimate_wait(position)

    await job_queue.put(job)

    if position == 1:
        await query.edit_message_text("Делаю видео... Пришлю сюда когда будет готово!")
    else:
        await query.edit_message_text(
            f"Принято! Ты {position}-й в очереди.\n"
            f"Примерное ожидание: ~{est} мин.\n\n"
            f"Я напишу когда начну делать твоё!"
        )

    return ConversationHandler.END

async def cancel(update: Update, context: ContextTypes.DEFAULT_TYPE) -> int:
    """Cancel current conversation."""
    context.user_data.clear()
    await update.message.reply_text(
        "Отменено. Нажми /start чтобы начать заново."
    )
    return ConversationHandler.END

@authorized
async def queue_status(update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
    """Show current queue status."""
    size = job_queue.qsize()
    if size == 0:
        await update.message.reply_text("Очередь пустая. Отправь /start чтобы сделать видео!")
    else:
        await update.message.reply_text(f"В очереди: {size} видео.")

# ==================== COMFYUI API ====================

def load_workflow():
    """Load workflow JSON template."""
    with open(WORKFLOW_PATH, "r") as f:
        return json.load(f)

def send_to_comfyui(workflow_json):
    """Send workflow to ComfyUI queue, return prompt_id."""
    data = json.dumps({"prompt": workflow_json}).encode("utf-8")
    req = urllib.request.Request(
        f"{COMFYUI_URL}/prompt",
        data=data,
        headers={"Content-Type": "application/json"},
    )
    resp = urllib.request.urlopen(req, timeout=30)
    return json.loads(resp.read()).get("prompt_id")

def poll_completion(prompt_id, timeout=3600):
    """Poll ComfyUI /history until prompt is done."""
    start_t = time.time()
    while time.time() - start_t < timeout:
        try:
            resp = urllib.request.urlopen(
                f"{COMFYUI_URL}/history/{prompt_id}", timeout=15
            )
            history = json.loads(resp.read())
            if prompt_id in history:
                return history[prompt_id]
        except Exception:
            pass
        time.sleep(5)
    return None

def find_output_from_history(history):
    """Extract output file path from ComfyUI history response."""
    for node_id, node_data in history.get("outputs", {}).items():
        for output_type in ("gifs", "images", "videos"):
            items = node_data.get(output_type, [])
            for item in items:
                filename = item.get("filename")
                if filename:
                    subfolder = item.get("subfolder", "")
                    path = os.path.join(COMFY_OUTPUT, subfolder, filename)
                    if os.path.exists(path):
                        return path
    return None

def find_latest_output():
    """Fallback: find most recently created file in ComfyUI output."""
    if not os.path.exists(COMFY_OUTPUT):
        return None
    files = [
        os.path.join(COMFY_OUTPUT, f)
        for f in os.listdir(COMFY_OUTPUT)
        if f.endswith((".mp4", ".webm", ".png", ".gif"))
    ]
    if not files:
        return None
    return max(files, key=os.path.getmtime)

# ==================== POST-PROCESSING ====================

def parse_fps(probe_output):
    """Safely parse fps from ffprobe output like '24/1' or '30000/1001'."""
    try:
        text = probe_output.strip()
        if "/" in text:
            num, den = text.split("/")
            return int(num) / int(den)
        return float(text)
    except (ValueError, ZeroDivisionError):
        return 24

def postprocess_video(input_path):
    """Upscale + interpolate + export. Returns path to final mp4."""
    os.makedirs(PROCESSED_DIR, exist_ok=True)
    basename = os.path.splitext(os.path.basename(input_path))[0]
    current = input_path

    if ENABLE_UPSCALE:
        logger.info("Upscaling...")
        frames_dir = os.path.join(PROCESSED_DIR, "frames_in")
        frames_up_dir = os.path.join(PROCESSED_DIR, "frames_up")
        os.makedirs(frames_dir, exist_ok=True)
        os.makedirs(frames_up_dir, exist_ok=True)

        subprocess.run(
            ["ffmpeg", "-y", "-i", current, "-qscale:v", "2",
             f"{frames_dir}/%06d.png", "-loglevel", "error"],
            check=True
        )

        result = subprocess.run(
            ["python", "-m", "realesrgan", "-i", frames_dir, "-o", frames_up_dir,
             "-n", "RealESRGAN_x4plus", "-s", "4", "--suffix", ""],
            capture_output=True
        )
        if result.returncode != 0:
            logger.warning("RealESRGAN failed, skipping upscale")
            frames_up_dir = frames_dir

        probe = subprocess.run(
            ["ffprobe", "-v", "error", "-select_streams", "v",
             "-of", "csv=p=0", "-show_entries", "stream=r_frame_rate", current],
            capture_output=True, text=True
        )
        fps = parse_fps(probe.stdout)

        upscaled = os.path.join(PROCESSED_DIR, f"{basename}_up.mp4")
        subprocess.run(
            ["ffmpeg", "-y", "-framerate", str(fps),
             "-i", f"{frames_up_dir}/%06d.png",
             "-c:v", "libx264", "-pix_fmt", "yuv420p", upscaled,
             "-loglevel", "error"],
            check=True
        )
        current = upscaled

        shutil.rmtree(frames_dir, ignore_errors=True)
        if frames_up_dir != frames_dir:
            shutil.rmtree(frames_up_dir, ignore_errors=True)

    if ENABLE_INTERPOLATION:
        logger.info("Interpolating...")
        interpolated = os.path.join(PROCESSED_DIR, f"{basename}_interp.mp4")
        subprocess.run(
            ["ffmpeg", "-y", "-i", current,
             "-vf", f"minterpolate=fps={TARGET_FPS}:mi_mode=mci:mc_mode=aobmc:me_mode=bidir:vsbmc=1",
             "-c:v", "libx264", "-pix_fmt", "yuv420p", interpolated,
             "-loglevel", "error"],
            check=True
        )
        current = interpolated

    final = os.path.join(PROCESSED_DIR, f"{basename}_final.mp4")
    subprocess.run(
        ["ffmpeg", "-y", "-i", current, "-an",
         "-c:v", "libx264", "-crf", "18", "-preset", "slow",
         "-pix_fmt", "yuv420p", final,
         "-loglevel", "error"],
        check=True
    )

    return final

def sync_file_to_drive(file_path):
    """Copy a single file to Google Drive."""
    if os.path.exists(DRIVE_OUTPUT):
        dst = os.path.join(DRIVE_OUTPUT, os.path.basename(file_path))
        shutil.copy2(file_path, dst)
        return True
    return False

# ==================== WORKER ====================

async def worker(app):
    """Process jobs from queue one by one."""
    logger.info("Worker started, waiting for jobs...")
    while True:
        job = await job_queue.get()
        start_time = time.time()

        try:
            await app.bot.send_message(
                chat_id=job["chat_id"],
                text="Начинаю делать твоё видео..."
            )

            # 1. Build workflow (deepcopy to avoid mutation issues)
            workflow = copy.deepcopy(load_workflow())

            for node_id, node in workflow.items():
                if not isinstance(node, dict):
                    continue
                inputs = node.get("inputs", {})
                for key in ["text", "prompt", "positive"]:
                    if key in inputs and isinstance(inputs[key], str):
                        inputs[key] = job["prompt"]

            for node_id, node in workflow.items():
                if not isinstance(node, dict):
                    continue
                inputs = node.get("inputs", {})
                if "width" in inputs and "height" in inputs:
                    inputs["width"] = job["width"]
                    inputs["height"] = job["height"]
                if "num_frames" in inputs:
                    inputs["num_frames"] = job["frames"]
                if "steps" in inputs and isinstance(inputs["steps"], (int, float)):
                    inputs["steps"] = DEFAULT_STEPS
                if "cfg" in inputs and isinstance(inputs["cfg"], (int, float)):
                    inputs["cfg"] = DEFAULT_CFG

            if job["mode"] == "mode_i2v" and job.get("image_path"):
                for node_id, node in workflow.items():
                    if not isinstance(node, dict):
                        continue
                    if node.get("class_type") in ("LoadImage", "Load Image"):
                        node["inputs"]["image"] = job["image_path"]

            # 2. Send to ComfyUI
            prompt_id = send_to_comfyui(workflow)
            if not prompt_id:
                raise RuntimeError("ComfyUI did not return prompt_id")

            # 3. Wait for completion
            history = await asyncio.get_event_loop().run_in_executor(
                None, poll_completion, prompt_id
            )
            if not history:
                raise RuntimeError("Generation timed out")

            # 4. Find output file (prefer history, fallback to latest)
            output_file = find_output_from_history(history) or find_latest_output()
            if not output_file:
                raise RuntimeError("No output file found")

            # 5. Post-process
            await app.bot.send_message(
                chat_id=job["chat_id"],
                text="Видео готово, улучшаю качество..."
            )
            final_path = await asyncio.get_event_loop().run_in_executor(
                None, postprocess_video, output_file
            )

            # 6. Send video to Telegram
            file_size_mb = os.path.getsize(final_path) / (1024 * 1024)

            if file_size_mb > 50:
                await app.bot.send_message(
                    chat_id=job["chat_id"],
                    text=f"Видео слишком большое для Telegram ({file_size_mb:.0f} MB).\n"
                         f"Сохранил на Google Drive!"
                )
            else:
                with open(final_path, "rb") as video_file:
                    await app.bot.send_video(
                        chat_id=job["chat_id"],
                        video=video_file,
                        caption="Готово!",
                        supports_streaming=True,
                    )

            # 7. Sync to Drive
            saved = sync_file_to_drive(final_path)
            drive_msg = "\nСохранено на Google Drive!" if saved else ""

            keyboard = [[InlineKeyboardButton("Сделать ещё", callback_data="restart")]]
            await app.bot.send_message(
                chat_id=job["chat_id"],
                text=f"Готово!{drive_msg}",
                reply_markup=InlineKeyboardMarkup(keyboard),
            )

            elapsed = (time.time() - start_time) / 60
            generation_times.append(elapsed)
            logger.info(f"Job done in {elapsed:.1f} min")

        except Exception as e:
            logger.error(f"Job failed: {e}")
            keyboard = [[InlineKeyboardButton("Попробовать снова", callback_data="restart")]]
            await app.bot.send_message(
                chat_id=job["chat_id"],
                text="Ой, что-то пошло не так.\nПопробуй ещё раз!",
                reply_markup=InlineKeyboardMarkup(keyboard),
            )

        finally:
            job_queue.task_done()

# ==================== LAUNCH ====================

async def post_init(app: Application) -> None:
    """Start worker after bot initialization."""
    asyncio.create_task(worker(app))
    logger.info("Bot is running! Open your bot in Telegram and press /start")

print("Starting Telegram bot...")
print("Press the STOP button in Colab to shut down.\n")

app = Application.builder().token(BOT_TOKEN).post_init(post_init).build()

conv_handler = ConversationHandler(
    entry_points=[
        CommandHandler("start", start),
        CallbackQueryHandler(restart_entry, pattern="^restart$"),
    ],
    states={
        CHOOSE_MODE: [CallbackQueryHandler(choose_mode, pattern="^mode_")],
        WAIT_PHOTO: [MessageHandler(filters.PHOTO, receive_photo)],
        WAIT_PROMPT: [MessageHandler(filters.TEXT & ~filters.COMMAND, receive_prompt)],
        CHOOSE_DURATION: [CallbackQueryHandler(choose_duration, pattern="^dur_")],
        CHOOSE_ORIENTATION: [CallbackQueryHandler(choose_orientation, pattern="^orient_")],
    },
    fallbacks=[CommandHandler("cancel", cancel)],
    per_message=False,
)

app.add_handler(conv_handler)
app.add_handler(CommandHandler("queue", queue_status))

app.run_polling(allowed_updates=Update.ALL_TYPES)